# 🪲 Exercise 05 — Debugging

**Day 2 · Open-ended**

Intentionally broken code. For each bug: **run** → **read the error** → **diagnose** → **fix**.

---

#### How to read a traceback

```
Traceback (most recent call last):      ← start from the BOTTOM
  File "app.py", line 42, in <module>
    result = df["Amenity"]
KeyError: 'Amenity'                     ← WHAT went wrong
```

**Last line first.** It tells you *what*. Then read upward for *where*.

In [ ]:
import pandas as pd
import numpy as np

# Setup: create a sample DataFrame (simulates cleaned OSM data)
sample_data = pd.DataFrame({
    "id": range(1, 101),
    "lat": np.random.uniform(59.30, 59.40, 100),
    "lon": np.random.uniform(17.95, 18.15, 100),
    "amenity": np.random.choice(["restaurant", "cafe", "bar", "fast_food", "pharmacy"], 100),
    "name": [f"Place {i}" if i % 3 != 0 else None for i in range(100)],
    "cuisine": np.random.choice(["italian", "japanese", "kebab", None], 100),
    "rating": np.random.choice([1, 2, 3, 4, 5, None], 100),
})

print(f"Sample data: {sample_data.shape}")
sample_data.head()

---

### 🟢 Level 1 — Easy

#### Bug 1: KeyError

In [ ]:
# 🐛 BROKEN — run this and read the error
counts = sample_data["Amenity"].value_counts()
print(counts)

In [ ]:
# ✏️ YOUR FIX: What's wrong? Write the corrected code below.
# Diagnosis: 
# Fix:

#### Bug 2: TypeError

In [ ]:
# 🐛 BROKEN — run this and read the error
count = len(sample_data)
label = "Total amenities: " + count
print(label)

In [ ]:
# ✏️ YOUR FIX:
# Diagnosis:
# Fix:

#### Bug 3: AttributeError

In [ ]:
# 🐛 BROKEN — run this and read the error
unique_types = sample_data["amenity"].Unique()
print(unique_types)

In [ ]:
# ✏️ YOUR FIX:
# Diagnosis:
# Fix:

---

### 🟡 Level 2 — Medium

These don't always produce errors — they just give wrong results.

#### Bug 4: Merge explosion

In [ ]:
# 🐛 BROKEN — runs without error, but the result is wrong

# Table 1: amenity prices
prices = pd.DataFrame({
    "amenity": ["restaurant", "cafe", "restaurant", "bar"],
    "avg_price": [250, 80, 400, 150]
})

# Table 2: amenity ratings
ratings = pd.DataFrame({
    "amenity": ["restaurant", "restaurant", "cafe", "bar"],
    "avg_rating": [4.2, 3.8, 4.5, 3.9]
})

merged = prices.merge(ratings, on="amenity")

print(f"Prices rows:  {len(prices)}")
print(f"Ratings rows: {len(ratings)}")
print(f"Merged rows:  {len(merged)}  ← Why is this MORE than either input?")
print()
merged

In [ ]:
# ✏️ YOUR FIX:
# Diagnosis: Why did the merge produce more rows?
#
# Fix: How would you prevent this? (Hint: make the key unique, or use validate="one_to_one")
#

#### Bug 5: Silent wrong result from groupby

In [ ]:
# 🐛 BROKEN — no error, but wrong answer

# Goal: average rating per amenity type
avg_rating = sample_data.groupby("amenity")["id"].mean()  # ← something's wrong here
print("Average 'rating' per amenity:")
print(avg_rating)

In [ ]:
# ✏️ YOUR FIX:
# Diagnosis: What column is actually being averaged?
#
# Fix:
#

#### Bug 6: Off-by-one in slicing

In [ ]:
# 🐛 BROKEN — off by one

top_types = sample_data["amenity"].value_counts()

# Attempt 1: using iloc (0-indexed, exclusive end)
print("iloc[0:4] gives us:")
print(top_types.iloc[0:4])  # How many results?

print("\nloc[0:4] gives us:")
# print(top_types.loc[0:4])  # ← This may crash or give unexpected results. Why?

In [ ]:
# ✏️ YOUR FIX:
# Diagnosis: What's the difference between .iloc and .loc? When is the end inclusive?
#
# Fix: Get exactly the top 5 using .head(), .iloc, or .loc — which is clearest?
#

---

### 🔴 Level 3 — Hard

Code runs fine — but produces wrong results or breaks later.

#### Bug 7: Import shadowing (thought exercise)

In [ ]:
# 🐛 SCENARIO: You have a file called requests.py in your project that contains:
#
#   # requests.py
#   BASE_URL = "https://overpass-api.de/api/interpreter"
#
# Then in your notebook you try:
#   import requests
#   response = requests.get(BASE_URL, params={"data": query})
#
# What happens? What error do you get?
# Why?

print("Think about it: what does 'import requests' load when there's")
print("a file called requests.py in the same directory?")
print()
print("Answer: Python loads YOUR requests.py instead of the real requests library.")
print("Your file doesn't have a .get() method → AttributeError.")
print()
print("Fix: NEVER name your files the same as a library you're importing.")
print("Bad names: requests.py, json.py, pandas.py, os.py")

#### Bug 8: SettingWithCopyWarning (silent data corruption)

In [ ]:
# 🐛 BROKEN — runs, but may not do what you think

# Get all restaurants
restaurants = sample_data[sample_data["amenity"] == "restaurant"]

# Try to add a new column to just the restaurants
try:
    restaurants["is_fancy"] = restaurants["cuisine"].apply(
        lambda x: True if x == "italian" else False
    )
    print(f"Added is_fancy column: {restaurants['is_fancy'].sum()} fancy restaurants")
except Exception as e:
    print(f"Error: {e}")

# Question: Did this modify sample_data too? Or just the slice?
print(f"\n'is_fancy' in sample_data? {'is_fancy' in sample_data.columns}")

In [ ]:
# ✏️ YOUR FIX:
# Diagnosis: What's the difference between a VIEW and a COPY in pandas?
#   restaurants = sample_data[sample_data["amenity"] == "restaurant"]  ← view or copy?
#
# Fix: How do you make sure you're working on an independent copy?
# Hint: .copy()
#

#### Bug 9: Duplicate directory names (thought exercise)

In [ ]:
# 🐛 SCENARIO: Think about it.

import sys
print("Python searches for modules in this order:")
for i, path in enumerate(sys.path[:5]):
    print(f"  {i}: {path}")
print("  ...")

print("\nThe FIRST match wins. If your current directory has src/utils.py,")
print("that's what gets imported — even if you meant the other one.")
print("\nFix: Use UNIQUE module names. Don't have two 'src/' or two 'utils.py'.")
print("Better: 'data_cleaning/' and 'data_processing/' instead of two 'src/' folders.")

---

### 🏆 Level 4 — Boss

No error, no warning — silently changes your results.

#### Bug 10: Silent dtype coercion

In [ ]:
# 🐛 BROKEN — no error, no warning, wrong dtype

# Simulate building levels data
buildings = pd.DataFrame({
    "building_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "levels":      [3, 5, 2, 8, 4, 1, 6, 3],  # All integers
})

print(f"Before: dtype = {buildings['levels'].dtype}")
print(f"Values: {buildings['levels'].tolist()}")

# Now add a row with missing levels
new_row = pd.DataFrame({"building_id": [9], "levels": [None]})
buildings = pd.concat([buildings, new_row], ignore_index=True)

print(f"\nAfter adding one NaN: dtype = {buildings['levels'].dtype}")
print(f"Values: {buildings['levels'].tolist()}")
print("\nNotice: 3 became 3.0, 5 became 5.0, etc.")
print("One NaN value forced the ENTIRE column to float64.")

In [ ]:
# ✏️ YOUR FIX:
# Diagnosis: Why does NaN force int → float?
#   (Hint: NaN is a float value. Standard int64 can't hold it.)
#
# Fix: Use pandas nullable integer type: "Int64" (capital I)
#
# buildings["levels"] = buildings["levels"].astype("Int64")
# print(buildings["levels"].tolist())  # [3, 5, 2, ...] — integers again!


---

| Level | Bug | Lesson |
|-------|-----|--------|
| 🟢 | KeyError | Column names are case-sensitive |
| 🟢 | TypeError | Can't concat str + int — use f-strings |
| 🟢 | AttributeError | Method names are case-sensitive |
| 🟡 | Merge explosion | Non-unique keys → row explosion |
| 🟡 | Wrong groupby column | Runs fine, averages the wrong thing |
| 🟡 | Off-by-one | `.iloc` excludes end, `.head(n)` is clearest |
| 🔴 | Import shadowing | Never name files after libraries |
| 🔴 | SettingWithCopy | Use `.copy()` when filtering |
| 🔴 | Duplicate dirs | Unique module names prevent confusion |
| 🏆 | Silent dtype coercion | NaN forces int→float; use `Int64` |

**The most dangerous bugs produce no errors.**

**Next →** [Exercise 06 — Build Streamlit App](06_build_streamlit_app.md)